# Data Science Capstone Project Introduction
This notebook runs through 3 stages: object detection, optical character recognition and parsing for NSW parking-related signs.  
This is for the Data Science Capstone project titled **_Australian Parking Sign Detection and Structured Information Extraction_**.  

By Harry Ngo 

Some sections assisted by AI

## 1. Stage - YOLOv8 Object Detection Model

This section walks through the object detection phase, including training, prediction and evaluation.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import time

In [ ]:
# training

start_time = time.perf_counter()
model = YOLO("yolov8n.pt")  # or yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt

results = model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=-1,
    workers=0,
    fliplr=0.0,  # disable horizontal flip
    flipud=0.0   # disable vertical flip
)

end_time = time.perf_counter()
execution_time_seconds = end_time - start_time
execution_time_minutes = execution_time_seconds / 60

print(f"Training execution time: {execution_time_seconds:.2f} seconds ({execution_time_minutes:.2f} minutes)")
print("Results saved to:", results.save_dir)
save_dir = results.save_dir
save_dir_path = Path(save_dir)
best_model_path = save_dir_path / "weights" / "best.pt"

### Validation Folder

In [ ]:
# validation and evaluation

best_model_path = "runs/detect/train/weights/best.pt"
model = YOLO(best_model_path)

results_val = model.val(data="dataset/data.yaml", split="val", device=-1, workers=0, save_json=True)
results_val.results_dict

In [ ]:
%matplotlib inline

save_dir_val = results_val.save_dir

# list of image filenames
image_files = [
    'BoxP_curve.png', 'BoxR_curve.png', 'BoxF1_curve.png',
    'BoxPR_curve.png', 'confusion_matrix_normalized.png', 'confusion_matrix.png'
]

fig, axes = plt.subplots(3, 2, figsize=(10, 15))
axes = axes.flatten()  # Flatten to 1D for easy iteration

for i, ax in enumerate(axes):
    img = Image.open(f'{save_dir_val}/{image_files[i]}')
    ax.imshow(img)
    ax.set_title(image_files[i])  # Use filename as title
    ax.axis('off')  # Hide axes

plt.tight_layout()
plt.show()

### Test Folder

Object detection on the Google Street View (GSV) `test/` folder.

In [ ]:
# test and evaluation

best_model_path = "runs/detect/train/weights/best.pt"
model = YOLO(best_model_path)

results_test = model.val(data="dataset/data.yaml", split="test", device=-1, workers=0, save_json=True)
results_test.results_dict

In [ ]:
%matplotlib inline

save_dir_test = results_test.save_dir

# list of image filenames
image_files = [
    'BoxP_curve.png', 'BoxR_curve.png', 'BoxF1_curve.png',
    'BoxPR_curve.png', 'confusion_matrix_normalized.png', 'confusion_matrix.png'
]

fig, axes = plt.subplots(3, 2, figsize=(10, 15))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img = Image.open(f'{save_dir_test}/{image_files[i]}')
    ax.imshow(img)
    ax.set_title(image_files[i])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Stage - Optical Character Recognition (OCR)

This section walks through the OCR phase, using PaddleOCR to extract the textual content of the `text-region-full` category, only if they are picked up by the previous object detections with at least 0.50 confidence score. This is done to both validation and test images.  
The process first crops the original images based on the predictions, processes each crop using PaddleOCR to output a .json file and then compares the
merged text with the ground truth text files to calculate character error rate (CER).

### Validation Images

In [ ]:
from pathlib import Path
from PIL import Image
import json
from collections import defaultdict

PRED_FILE = Path("runs/detect/val/predictions.json")
IMG_DIR = Path("dataset/images/val")
OCR_DIR = Path("ocr")

CROP_DIR = OCR_DIR / "crops"
GT_DIR = OCR_DIR / "gt_text"
META_DIR = OCR_DIR / "meta"

# create dirs if missing
for d in [CROP_DIR, GT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SCORE_THRESH = 0.5
TEXT_REGION_CLASS = 3

# helper: find image file with any valid extension
def find_image(stem):
    for ext in [".jpg", ".jpeg", ".png"]:
        p = IMG_DIR / f"{stem}{ext}"
        if p.exists():
            return p
    return None

# load model predictions
with open(PRED_FILE) as f:
    predictions = json.load(f)

# fix category indexing to be aligned: move from 1–N → 0–N-1
for p in predictions:
    p["category_id"] -= 1

# keep only text-region detections above threshold
filtered_preds = [
    p for p in predictions
    if p["category_id"] == TEXT_REGION_CLASS and p["score"] >= SCORE_THRESH
]

# group detections by image
grouped = defaultdict(list)
for p in filtered_preds:
    grouped[p["image_id"]].append(p)

print(f"Found {len(grouped)} images with text regions.")

# padding in pixels
PAD = 10

# process each image
for image_id, boxes in grouped.items():
    img_path = find_image(image_id)
    if not img_path:
        print(f"Image not found: {image_id}")
        continue

    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size

    for i, p in enumerate(boxes):

        # bbox values from YOLO json format
        x, y, w, h = p["bbox"]

        # clamp to image boundaries
        x1 = max(0, int(x - PAD))
        y1 = max(0, int(y - PAD))
        x2 = min(img_w, int(x + w + PAD))
        y2 = min(img_h, int(y + h + PAD))

        if x2 <= x1 or y2 <= y1:
            print(f"Invalid crop for {image_id}, skip.")
            continue

        # crop extraction
        crop = img.crop((x1, y1, x2, y2))

        # optional: pad with white if bbox hits image edge
        new_crop = Image.new("RGB", (x2 - x1, y2 - y1), (255, 255, 255))
        new_crop.paste(crop, (0, 0))
        crop = new_crop
        
        crop_name = f"{image_id}_crop{i}.png"
        crop_path = CROP_DIR / crop_name
        crop.save(crop_path)

        # create empty GT txt file (placeholder for manual correction)
        gt_path = GT_DIR / f"{image_id}_crop{i}.txt"
        if not gt_path.exists():
            open(gt_path, "w").close()

        # metadata file
        meta_path = META_DIR / f"{image_id}_crop{i}.json"
        meta = {
            "image_id": image_id,
            "crop_id": i,
            "bbox": [x1, y1, x2, y2],
            "score": float(p["score"]),
            "crop_file": crop_name,
            "gt_file": gt_path.name,
        }
        with open(meta_path, "w") as jf:
            json.dump(meta, jf, indent=2)

        print(f"Saved crop: {crop_name}")

print("Cropping complete!")
print(f"Crops saved to: {CROP_DIR}")
print(f"Metadata saved to: {META_DIR}")
print(f"GT files saved to: {GT_DIR}")

### Test Images

In [ ]:
from pathlib import Path
from PIL import Image
import json
from collections import defaultdict

PRED_FILE = Path("runs/detect/val2/predictions.json")
IMG_DIR = Path("dataset/images/test")
OCR_DIR = Path("ocr_test")

CROP_DIR = OCR_DIR / "crops"
GT_DIR = OCR_DIR / "gt_text"
META_DIR = OCR_DIR / "meta"

# create dirs if missing
for d in [CROP_DIR, GT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SCORE_THRESH = 0.5
TEXT_REGION_CLASS = 3

# helper: find image file with any valid extension
def find_image(stem):
    for ext in [".jpg", ".jpeg", ".png"]:
        p = IMG_DIR / f"{stem}{ext}"
        if p.exists():
            return p
    return None

# load model predictions
with open(PRED_FILE) as f:
    predictions = json.load(f)

# fix category indexing to be aligned: move from 1–N → 0–N-1
for p in predictions:
    p["category_id"] -= 1

# keep only text-region detections above threshold
filtered_preds = [
    p for p in predictions
    if p["category_id"] == TEXT_REGION_CLASS and p["score"] >= SCORE_THRESH
]

# group detections by image
grouped = defaultdict(list)
for p in filtered_preds:
    grouped[p["image_id"]].append(p)

print(f"Found {len(grouped)} images with text regions.")

# padding in pixels
PAD = 10

# process each image
for image_id, boxes in grouped.items():
    img_path = find_image(image_id)
    if not img_path:
        print(f"Image not found: {image_id}")
        continue

    img = Image.open(img_path).convert("RGB")
    img_w, img_h = img.size

    for i, p in enumerate(boxes):

        # bbox values from YOLO json format
        x, y, w, h = p["bbox"]

        # clamp to image boundaries
        x1 = max(0, int(x - PAD))
        y1 = max(0, int(y - PAD))
        x2 = min(img_w, int(x + w + PAD))
        y2 = min(img_h, int(y + h + PAD))

        if x2 <= x1 or y2 <= y1:
            print(f"Invalid crop for {image_id}, skip.")
            continue

        # crop extraction
        crop = img.crop((x1, y1, x2, y2))

        # optional: pad with white if bbox hits image edge
        new_crop = Image.new("RGB", (x2 - x1, y2 - y1), (255, 255, 255))
        new_crop.paste(crop, (0, 0))
        crop = new_crop
        
        crop_name = f"{image_id}_crop{i}.png"
        crop_path = CROP_DIR / crop_name
        crop.save(crop_path)

        # create empty GT txt file (placeholder for manual correction)
        gt_path = GT_DIR / f"{image_id}_crop{i}.txt"
        if not gt_path.exists():
            open(gt_path, "w").close()

        # metadata file
        meta_path = META_DIR / f"{image_id}_crop{i}.json"
        meta = {
            "image_id": image_id,
            "crop_id": i,
            "bbox": [x1, y1, x2, y2],
            "score": float(p["score"]),
            "crop_file": crop_name,
            "gt_file": gt_path.name,
        }
        with open(meta_path, "w") as jf:
            json.dump(meta, jf, indent=2)

        print(f"Saved crop: {crop_name}")

print("Cropping complete!")
print(f"Crops saved to: {CROP_DIR}")
print(f"Metadata saved to: {META_DIR}")
print(f"GT files saved to: {GT_DIR}")

In [ ]:
from paddleocr import PaddleOCR
from PIL import Image, ImageEnhance
import numpy as np
from pathlib import Path
import json
import time
import paddle
import random

SEED = 42
paddle.seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

ocr = PaddleOCR(
    lang="en",
    use_textline_orientation=True
)


def safe_json_dump(data, fpath):
    def convert(o):
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (np.float32, np.float64)):
            return float(o)
        if isinstance(o, (np.int32, np.int64)):
            return int(o)
        return o

    with open(fpath, "w", encoding="utf-8") as f:
        json.dump(data, f, default=convert, indent=2, ensure_ascii=False)


def enhance_image(img):
    """
    Only grayscale -> contrast -> sharpness -> RGB
    Avoid EDGE_ENHANCE to reduce noise.
    """
    img_e = img.convert("L")
    img_e = ImageEnhance.Contrast(img_e).enhance(1.8)
    img_e = ImageEnhance.Sharpness(img_e).enhance(1.5)
    return img_e.convert("RGB")


def load_image(img_path, resize_width=None):
    img = Image.open(img_path).convert("RGB")
    if resize_width and img.width > resize_width:
        new_h = int(img.height * resize_width / img.width)
        img = img.resize((resize_width, new_h))
    return img, np.array(img)


def run_ocr(img_np, conf_thresh=0.3):
    results = ocr.predict(img_np)
    if not results or not isinstance(results, list):
        return [], [], []

    page = results[0] if isinstance(results[0], dict) else results[0]
    rec_texts = page.get("rec_texts", [])
    rec_scores = page.get("rec_scores", [])
    rec_boxes = page.get("rec_boxes", [])

    texts, scores, boxes = [], [], []
    for t, s, b in zip(rec_texts, rec_scores, rec_boxes):
        if s >= conf_thresh:
            texts.append(t)
            scores.append(float(s))
            boxes.append(b)
    return texts, scores, boxes


def scoring_metric(scores, texts):
    if not scores:
        return 0.0
    avg_conf = float(np.mean(scores))
    count = len(scores)
    return avg_conf + 0.05 * count  # primary weight on confidence


def run_paddleocr_dual(img_path, conf_thresh=0.3, resize_width=None, enhanced_threshold=0.01):
    img_orig_pil, img_orig_np = load_image(img_path, resize_width)

    img_enh_pil = enhance_image(img_orig_pil)
    img_enh_np = np.array(img_enh_pil)

    # original OCR
    texts_o, scores_o, boxes_o = run_ocr(img_orig_np, conf_thresh)
    metric_o = scoring_metric(scores_o, texts_o)

    # enhanced OCR
    texts_e, scores_e, boxes_e = run_ocr(img_enh_np, conf_thresh)
    metric_e = scoring_metric(scores_e, texts_e)

    # choose the best
    chosen = "original"
    if metric_e >= (metric_o + enhanced_threshold):
        chosen = "enhanced"

    if chosen == "enhanced":
        texts, scores, boxes = texts_e, scores_e, boxes_e
    else:
        texts, scores, boxes = texts_o, scores_o, boxes_o

    merged_text = " ".join(texts)
    return {
        "texts": texts,
        "scores": scores,
        "merged_text": merged_text,
        "which_version": chosen,
        "metric_original": metric_o,
        "metric_enhanced": metric_e
    }


def process_folder(input_dir, output_json, conf_thresh=0.3, resize_width=None):
    input_dir = Path(input_dir)
    results_dict = {}
    start = time.time()

    img_paths = sorted(
        [p for p in input_dir.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]],
        key=lambda p: p.name.lower()
    )

    total = len(img_paths)
    for idx, img_path in enumerate(img_paths, 1):
        print(f"[{idx}/{total}] Processing: {img_path.name}")
        res = run_paddleocr_dual(img_path, conf_thresh=conf_thresh, resize_width=resize_width)
        results_dict[img_path.name] = res
        print(f" -> chosen: {res['which_version']}, detections: {len(res['texts'])}, "
              f"metric_orig={res['metric_original']:.3f}, metric_enh={res['metric_enhanced']:.3f}")

        if idx % 20 == 0:
            print(f"Processed {idx} images...")

    elapsed = time.time() - start
    print(f"Completed {len(results_dict)} images in {elapsed:.1f}s")
    safe_json_dump(results_dict, output_json)

In [ ]:
# example usage:
process_folder("ocr/crops", "ocr/results_val.json", conf_thresh=0.3, resize_width=600)
process_folder("ocr_test/crops", "ocr_test/results_test.json", conf_thresh=0.3, resize_width=600)

### Character Accuracy

$$ \text{CharAcc} = \frac{\text{\# of correctly detected characters in OCR (from GT)}}{\text{\# of unique characters in GT}} $$

- Count how many characters from the ground truth appear in the OCR output (ignoring order, duplicates optional).
- Divide by total GT characters.
- Cap at 1.0.

That gives a metric closer to “did OCR detect all the right alphanumeric characters somewhere in the text”, regardless of order.

In [ ]:
import json
import re
from pathlib import Path
import numpy as np
from collections import Counter


def normalize_text(text):
    """Uppercase, remove spaces/symbols/punctuation for fair comparison."""
    text = text.upper()
    text = re.sub(r"[^A-Z0-9]", "", text)  # keep only alphanumeric
    return text


def compute_char_accuracy(gt_text, pred_text):
    """Compute order-independent character overlap accuracy."""
    gt = normalize_text(gt_text)
    pred = normalize_text(pred_text)
    if len(gt) == 0:
        return None

    gt_count = Counter(gt)
    pred_count = Counter(pred)

    # count characters correctly detected (ignoring order)
    correct = sum(min(gt_count[ch], pred_count.get(ch, 0)) for ch in gt_count)
    return correct / len(gt)

def evaluate_ocr_characc(result_json_path, gt_dir):
    """Compute mean character accuracy across all valid samples."""
    result_path = Path(result_json_path)
    gt_path = Path(gt_dir)
    with open(result_path, "r", encoding="utf-8") as f:
        ocr_data = json.load(f)

    acc_scores = []
    skipped_empty_gt = 0
    skipped_missing_gt = 0

    for img_name, info in ocr_data.items():
        gt_file = gt_path / f"{Path(img_name).stem}.txt"
        if not gt_file.exists():
            skipped_missing_gt += 1
            continue

        gt_text = gt_file.read_text(encoding="utf-8").strip()
        if not gt_text:
            skipped_empty_gt += 1
            continue

        pred_text = info.get("merged_text", "").strip()
        acc = compute_char_accuracy(gt_text, pred_text)
        if acc is not None:
            acc_scores.append(acc)

    mean_acc = np.mean(acc_scores) if acc_scores else None
    print(f"Processed {len(acc_scores)} valid samples.")
    print(f"Skipped {skipped_missing_gt} missing GT and {skipped_empty_gt} empty GT files.")
    return mean_acc

In [ ]:
val_results_path = "ocr/results_val.json"
val_gt_dir = "ocr/gt_text"
test_results_path = "ocr_test/results_test.json"
test_gt_dir = "ocr_test/gt_text"

print("Evaluating validation OCR performance")
val_acc = evaluate_ocr_characc(val_results_path, val_gt_dir)

print("\nEvaluating test OCR performance")
test_acc = evaluate_ocr_characc(test_results_path, test_gt_dir)

print(f"\nValidation Character Accuracy: {val_acc:.4f}")
print(f"Test Character Accuracy: {test_acc:.4f}")

| Dataset        | Mean Char Accuracy | Interpretation                                                                                                                                          |
| -------------- | ------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Validation** | 0.8686 (~87%)      | On average, 87% of the characters in the ground truth are present somewhere in the OCR output. |
| **Test**       | 0.7760 (~78%)      | Slightly worse, likely due to unseen signs, more complex layouts, or angled text. |


### Merging OCR Results with Ground Truth Text

In [ ]:
import json
from pathlib import Path

def attach_gt_to_results(
    ocr_dir: str,
    results_filename: str,
    output_filename: str = None,
    gt_subdir: str = "gt_text"
):
    """
    ocr_dir: folder containing results JSON and gt_text/ subfolder
    results_filename: e.g. 'results_val.json'
    output_filename: e.g. 'results_val_with_gt.json' (if None, auto-add '_with_gt')
    gt_subdir: relative folder for ground truth txt files
    """
    ocr_dir = Path(ocr_dir)
    results_path = ocr_dir / results_filename
    gt_dir = ocr_dir / gt_subdir

    if output_filename is None:
        output_filename = results_path.stem + "_with_gt.json"
    output_path = ocr_dir / output_filename

    # load OCR results
    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    missing_gt = []
    count_with_gt = 0

    for img_name, entry in results.items():
        # 'IMG_012_crop0.png' -> 'IMG_012_crop0'
        stem = Path(img_name).stem
        gt_path = gt_dir / f"{stem}.txt"

        if gt_path.exists():
            with open(gt_path, "r", encoding="utf-8") as gf:
                lines = [ln.strip() for ln in gf.readlines() if ln.strip()]

            # store GT as list + merged string
            entry["gt_lines"] = lines
            entry["gt_merged"] = " ".join(lines)
            count_with_gt += 1
        else:
            # no GT file found – keep track if you want to inspect later
            missing_gt.append(stem)

    # save augmented JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Attached GT to {count_with_gt} entries.")
    if missing_gt:
        print(f"Warning: {len(missing_gt)} entries had no GT txt file, e.g.: {missing_gt[:5]}")
    print(f"Saved combined JSON to: {output_path}")

In [ ]:
# run for validation
attach_gt_to_results(
    ocr_dir="ocr",
    results_filename="results_val.json",
    output_filename="results_val_with_gt.json"
)

# run for test
attach_gt_to_results(
    ocr_dir="ocr_test",
    results_filename="results_test.json",
    output_filename="results_test_with_gt.json"
)

## 3. Stage - Structured Information Parsing

This section walks through the structured information parsing phase, which includes completing normalisation and correcting spelling errors using domain vocabulary to standardise the output.

### Normalisation and Correction of OCR Output

In [ ]:
import re
from difflib import get_close_matches

# domain vocabulary
KEYWORDS = {
    "STOPPING", "PARKING", "LOADING", "ZONE", "ONLY",
    "TAXIS", "EXCEPTED", "PERMIT", "HOLDERS", "CLEARWAY",
    "BUS", "AREA", "METER", "TICKET", "PARALLEL", "ANGLE",
    "MINUTE", "MINUTES", "HOUR", "HOURS", "DISABLED",
    "PARK", "KEEP", "MIDNIGHT", "NIGHT", "SCHOOL",
    "WORKS", "REAR", "KERB", "VEHICLES", "UNDER",
    "AUTHORISED", "CAR", "SHARE", "MAIL", "SPECIAL", "EVENT",
    "NORMAL", "RESTRICTIONS", "APPLY", "ALL", "OTHER",
    "TIMES", "LIMIT", "MOTORCYCLE", "BUSES", "AUST", "POST"
}

DAY_TOKENS = {
    "MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN",
    "MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY",
    "SATURDAY", "SUNDAY",
    "PUBLIC", "HOLIDAYS", "HOLIDAY",
    "MON-FRI", "MON-SAT", "SAT-SUN", "MON-SUN", "MON-THU",
    "SAT&SUN", "THURS"
}

TIME_LIMIT_TOKENS = {
    "1P", "2P", "4P", "P",
    "15", "30", "60",
    "15MIN", "30MIN", "60MIN",
    "1", "5"
}

DOMAIN_VOCAB = KEYWORDS | DAY_TOKENS | TIME_LIMIT_TOKENS

DOMAIN_VOCAB |= {"-", "AM", "PM", "&"}

# obvious single-char OCR confusions (deliberately *not* including 'I' here)
COMMON_ERRORS = {
    'l': '-',
    'O': '0',
    'S': '5',
    'B': '8',
}

DAYS_ORDER = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']

DAY_CANONICAL = {
    # Monday
    'MON': 'MON',
    'MONDAY': 'MON',
    'MONDAYS': 'MON',

    # Tuesday
    'TUE': 'TUE',
    'TUES': 'TUE',
    'TUESDAY': 'TUE',
    'TUESDAYS': 'TUE',

    # Wednesday
    'WED': 'WED',
    'WEDS': 'WED',
    'WEDNESDAY': 'WED',
    'WEDNESDAYS': 'WED',

    # Thursday
    'THU': 'THU',
    'THUR': 'THU',
    'THURS': 'THU',
    'THURSDAY': 'THU',
    'THURSDAYS': 'THU',

    # Friday
    'FRI': 'FRI',
    'FRIDAY': 'FRI',
    'FRIDAYS': 'FRI',

    # Saturday
    'SAT': 'SAT',
    'SATURDAY': 'SAT',
    'SATURDAYS': 'SAT',

    # Sunday
    'SUN': 'SUN',
    'SUNDAY': 'SUN',
    'SUNDAYS': 'SUN'
}

TIME_MINUTE_VALUES = {"00", "15", "30", "45"}
MERIDIEMS = {"AM", "PM"}


def fix_numeric_ocr(tok: str) -> str:
    """
    Fix common OCR confusions inside numeric-like tokens:
    - '1O' -> '10', '70' vs '7O', etc.
    - 'I' in the middle of digits -> '1'
    Only applied if the token has at least one digit.
    """
    if any(ch.isdigit() for ch in tok):
        # O / Q mistaken for 0 when adjacent to digits
        tok = re.sub(r"(?<=\d)[OQ](?=\D|\b)", "0", tok)
        tok = re.sub(r"(?<=\d)[OQ](?=\d)", "0", tok)
        # I mistaken for 1 between digits
        tok = re.sub(r"(?<=\d)I(?=\d)", "1", tok)
    return tok


def pre_normalize_tokens(text: str) -> list[str]:
    """
    - Uppercase
    - Normalise '&' to '-' between day names (SAT &SUN -> SAT-SUN)
    - Fix numeric OCR confusions inside tokens
    - Fix AU/PU -> AM/PM for misread meridiems, even when stuck into a time range (7AU-10PU)
    - handle MID NIGHT -> MIDNIGHT and normalise NOON
    """
    if not text:
        return []

    t = text.upper()

    # replace " & " with "-" (SAT &SUN -> SAT-SUN)
    t = re.sub(r"\s*&\s*", "-", t)

    raw_tokens = t.split()

    tokens: list[str] = []
    for tok in raw_tokens:
        tok = fix_numeric_ocr(tok)

        # fix AU/PU inside tokens (e.g. 7AU-10PU -> 7AM-10PM)
        tok = tok.replace("AU", "AM").replace("PU", "PM").replace("FR", "FRI").replace("BUG", "BUS")

        # fix "7M-4" -> "7AM-4"
        m = re.match(r'^(\d+)M-(\d+)$', tok)
        if m:
            start, end = m.groups()
            tok = f"{start}AM-{end}"

        # common OCR confusions of 1P
        if len(tok) == 2 and tok[1] == 'P' and tok[0] in {'T', 'I', 'L'}:
            tok = "1P"

        tokens.append(tok)

    # merge "MID NIGHT"
    merged: list[str] = []
    i = 0
    while i < len(tokens):
        current = tokens[i].upper()

        # look ahead for "MID" + "NIGHT"
        if i + 1 < len(tokens):
            nxt = tokens[i+1].upper()
            if current == "MID" and nxt == "NIGHT":
                merged.append("MIDNIGHT")
                i += 2
                continue

        merged.append(tokens[i])
        i += 1

    # normalise "NOON"
    final_tokens = [
        "NOON" if tok.upper() == "NOON" else tok
        for tok in merged
    ]

    return final_tokens


def canonical_day(token: str) -> str | None:
    """
    Map any day-like token (MON, MONDAY, THURS, THURSDAY, etc.)
    to its canonical 3-letter form (MON, TUE, WED, THU, FRI, SAT, SUN).
    Returns None if not recognised.
    """
    t = re.sub(r'[^A-Z]', '', token.upper())
    return DAY_CANONICAL.get(t)


def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )
    return dp[-1][-1]


def normalize_time(token: str) -> str:
    """
    Normalize compacted times like '830' -> '8:30' or '0600' -> '6:00'
    but leave 2-digit numbers (15, 30, 60) alone.
    """
    if re.match(r'^\d{3,4}$', token):
        if len(token) == 3:
            # 830 -> 8:30
            return f"{int(token[0])}:{token[1:]}"
        else:
            # 0600 or 1800 -> 6:00 or 18:00
            h = int(token[:2])
            return f"{h}:{token[2:]}"
    return token


def is_time_token(token: str) -> bool:
    # simple "H" or "H:MM" shape, H 1–12
    m = re.match(r'^(\d{1,2})(:\d{2})?$', token)
    if not m:
        return False
    h = int(m.group(1))
    return 1 <= h <= 12


def preprocess_text(text: str) -> str:
    """
    Uppercase + normalize AM/PM and dash characters before token-level logic.
    """
    t = text.upper()
    # normalize weird dashes
    t = re.sub(r"[–—−]", "-", t)
    # normalize AM/PM variants
    t = re.sub(r"\bA\.?M\.?\b", "AM", t)
    t = re.sub(r"\bP\.?M\.?\b", "PM", t)
    t = re.sub(r"\bA M\b", "AM", t)
    t = re.sub(r"\bP M\b", "PM", t)
    return t


def is_hour(tok: str) -> bool:
    return tok.isdigit() and 1 <= int(tok) <= 12


def is_time_range(tok: str) -> bool:
    # e.g. "7AM-10", "7-10PM", "8:30AM-6:00PM"
    return bool(re.match(r'^\d{1,2}(:\d{2})?(AM|PM)?-\d{1,2}(:\d{2})?(AM|PM)?$', tok))


def clean_ocr_text(text: str,
                   vocab: set = DOMAIN_VOCAB,
                   max_dist: int = 1) -> dict:
    """
    Main cleaning pipeline:

    1) Pre-normalize AM/PM and dashes.
    2) Tokenize + normalize compact times.
    3) Merge split "8 30" -> "8:30".
    4) Handle '8 6 PM AM' -> '8AM-6PM'.
    5) Handle '8:30 - 6 00 AM PM' -> '8:30AM-6:00PM'.
    6) Attach loose AM/PM to preceding time.
    7) Fix 'FRI I MON' -> 'MON-FRI' (with correct ordering).
    8) Spell-correct only reasonably safe tokens.
    """

    raw = text
    text = preprocess_text(text)

    # --- 1. Tokenize & normalize compact times ---
    tokens = pre_normalize_tokens(text)
    new_tokens = []
    for t in tokens:
        # handle compact "830AM" / "0630PM" style tokens
        m = re.match(r'^(\d{3,4})(AM|PM)$', t)
        if m:
            num, mer = m.group(1), m.group(2)
            num_norm = normalize_time(num)  # "830" -> "8:30"
            t = num_norm + mer              # "8:30AM"

        # then run normal compact-time normalisation on pure numeric tokens
        t = normalize_time(t)
        
        # split "2P-10" or "1/2P-8" into ['2P', '-', '10'] ---
        m_p = re.match(r'^(\d+(?:/\d+)?P)-(.*)$', t)
        if m_p and m_p.group(2):
            new_tokens.extend([m_p.group(1), '-', m_p.group(2)])
            continue
        
        # split attached dashes like "-FRI" or "MON-"
        if t.startswith('-') and len(t) > 1:
            new_tokens.append('-')
            new_tokens.append(t[1:])
        elif t.endswith('-') and len(t) > 1:
            new_tokens.append(t[:-1])
            new_tokens.append('-')
        else:
            new_tokens.append(t)
    tokens = new_tokens

    normalized_tokens = []
    corrections = {}

    i = 0
    while i < len(tokens):
        token = tokens[i]
        original = token
        
        # --- SPECIAL CASE: "H1 H2 AM - PM" -> "H1AM-H2PM" ---
        if i + 4 < len(tokens):
            h1, h2, mer1, dash, mer2 = tokens[i:i+5]
            if (
                is_hour(h1) and is_hour(h2) and
                mer1 == "AM" and dash == "-" and mer2 == "PM"
            ):
                normalized_tokens.append(f"{h1}AM-{h2}PM")
                i += 5
                continue
                
        # --- SPECIAL CASE 1b: "H1 H2 - AM PM" -> "H1AM-H2PM" ---
        if i + 5 < len(tokens):
            h1, h2, dash, mer1, mer2 = tokens[i:i+5]
            if (
                is_hour(h1) and is_hour(h2) and
                dash == '-' and
                mer1 in MERIDIEMS and mer2 in MERIDIEMS
            ):
                # force AM start, PM end
                normalized_tokens.append(f"{h1}AM-{h2}PM")
                i += 5
                continue
        
        # --- SPECIAL CASE 2: "H1 M1 H2 M2 AM PM" -> "H1:M1AM-H2:M2PM" ---
        if i + 5 < len(tokens):
            h1, m1, h2, m2, mer1, mer2 = tokens[i:i+6]
            if (
                h1.isdigit() and h2.isdigit() and
                m1.isdigit() and m2.isdigit() and
                m1 in TIME_MINUTE_VALUES and m2 in TIME_MINUTE_VALUES and
                mer1 in MERIDIEMS and mer2 in MERIDIEMS
            ):
                normalized_tokens.append(
                    f"{int(h1)}:{m1}{mer1}-{int(h2)}:{m2}{mer2}"
                )
                i += 6
                continue

        # common 1-char OCR fixes (but note: no 'I' here!)
        if len(token) == 1 and token in COMMON_ERRORS:
            token = COMMON_ERRORS[token]
            corrections[original] = token

        # merge split minutes: '8' '30' -> '8:30' (only if hour <13)
        if (
            token.isdigit() and 1 <= int(token) <= 12 and
            i + 1 < len(tokens) and
            tokens[i + 1].isdigit() and
            len(tokens[i + 1]) == 2 and
            tokens[i + 1] in TIME_MINUTE_VALUES
        ):
            token = f"{token}:{tokens[i + 1]}"
            i += 1

        # special case: H H PM AM or H H AM PM -> H AM - H PM
        if (token.isdigit() or is_time_token(token)) and i + 3 < len(tokens) \
                and (tokens[i + 1].isdigit() or is_time_token(tokens[i + 1])) \
                and tokens[i + 2] in {'AM', 'PM'} \
                and tokens[i + 3] in {'AM', 'PM'}:
            h1 = token
            h2 = tokens[i + 1]
            first, second = tokens[i + 2], tokens[i + 3]
            # we want start AM, end PM regardless of noisy order
            if first == 'AM' and second == 'PM':
                start_mer, end_mer = 'AM', 'PM'
            elif first == 'PM' and second == 'AM':
                start_mer, end_mer = 'AM', 'PM'
            else:
                # worst case, just sort so AM is first
                start_mer, end_mer = sorted([first, second])
            range_str = f"{h1}{start_mer}-{h2}{end_mer}"
            normalized_tokens.append(range_str)
            i += 4
            continue

        # normal time range: '8:30' '-' '6' [ '00' ] [AM PM]
        if is_time_token(token) and i + 1 < len(tokens) and tokens[i + 1] == '-' \
                and i + 2 < len(tokens) and (tokens[i + 2].isdigit() or is_time_token(tokens[i + 2])):
            start = token
            end = tokens[i + 2]
            j = i + 3

            # end minutes: e.g. '6' '00' -> '6:00'
            if j < len(tokens) and tokens[j].isdigit() and len(tokens[j]) == 2:
                end = f"{end}:{tokens[j]}"
                j += 1

            # AM/PM sequence after the range
            mer1 = mer2 = None
            if j < len(tokens) and tokens[j] in {'AM', 'PM'}:
                mer1 = tokens[j]
                j += 1
            if j < len(tokens) and tokens[j] in {'AM', 'PM'}:
                mer2 = tokens[j]
                j += 1

            if mer1 and mer2:
                start = start + mer1
                end = end + mer2

            normalized_tokens.append(f"{start}-{end}")
            i = j
            continue

        # otherwise: attach AM/PM directly to preceding time
        if is_time_token(token) and i + 1 < len(tokens) and tokens[i + 1] in {'AM', 'PM'}:
            token += tokens[i + 1]
            i += 1

        normalized_tokens.append(token)
        i += 1

    # --- 2. Attach loose AM/PM to previous time (if any) ---
    q = 0
    while q < len(normalized_tokens):
        if normalized_tokens[q] in {'AM', 'PM'}:
            if q > 0:
                prev = normalized_tokens[q - 1]
                # strip any existing AM/PM for the check
                base = re.sub(r'(AM|PM)$', '', prev)
                if is_time_token(base):
                    normalized_tokens[q - 1] = prev + normalized_tokens[q]
                    del normalized_tokens[q]
                    continue
        q += 1

    # --- after the "attach loose AM/PM to previous time" loop ---
    # Attach trailing AM/PM directly to a time range token:
    # - "8AM-10", "PM"   -> "8AM-10PM"
    # - "6-10", "AM"     -> "6AM-10AM"
    j = 0
    while j < len(normalized_tokens) - 1:
        tr, mer = normalized_tokens[j], normalized_tokens[j + 1]
        if is_time_range(tr) and mer in MERIDIEMS:
            start, end = tr.split('-')
            start_has_mer = start.endswith(tuple(MERIDIEMS))
            end_has_mer = end.endswith(tuple(MERIDIEMS))

            if start_has_mer and not end_has_mer:
                # "8AM-10", "PM" -> "8AM-10PM"
                end = end + mer
            elif not start_has_mer and not end_has_mer:
                # "6-10", "AM" -> "6AM-10AM"
                start = start + mer
                end = end + mer
            else:
                # both already have meridiems; nothing to do
                j += 1
                continue

            normalized_tokens[j] = f"{start}-{end}"
            del normalized_tokens[j + 1]
            continue

        j += 1
    
    # post-process X: attach trailing AM/PM after a day-range to the END of a range
    j = 0
    while j < len(normalized_tokens) - 2:
        tr, maybe_day, mer = normalized_tokens[j:j+3]
        if is_time_range(tr) and mer in MERIDIEMS:
            start, end = tr.split('-')
            has_start_mer = start.endswith(tuple(MERIDIEMS))
            has_end_mer = end.endswith(tuple(MERIDIEMS))
            if has_start_mer and not has_end_mer:
                # e.g. "7AM-10", "MON-FRI", "PM" -> "7AM-10PM", "MON-FRI"
                end = end + mer
                normalized_tokens[j] = f"{start}-{end}"
                del normalized_tokens[j+2]
                continue
        j += 1

    # --- 2b. Join consecutive time tokens into ranges: "8:30AM 6:30PM" -> "8:30AM-6:30PM" ---
    joined_times = []
    idx = 0
    while idx < len(normalized_tokens):
        tok = normalized_tokens[idx]

        # helper to check time-like even with AM/PM attached
        def is_time_with_meridiem(s: str) -> bool:
            base = re.sub(r'(AM|PM)$', '', s)
            return is_time_token(base)

        if idx + 1 < len(normalized_tokens) \
                and is_time_with_meridiem(tok) \
                and is_time_with_meridiem(normalized_tokens[idx + 1]):
            # example: "8:30AM", "6:30PM" -> "8:30AM-6:30PM"
            joined_times.append(f"{tok}-{normalized_tokens[idx + 1]}")
            idx += 2
        else:
            joined_times.append(tok)
            idx += 1

    normalized_tokens = joined_times
    
    # --- 2c. Heuristic: "8:30-6 PM" -> "8:30AM-6PM", "830-4 PM" -> "8:30AM-4PM" ---
    j = 0
    while j < len(normalized_tokens) - 1:
        tr, mer = normalized_tokens[j], normalized_tokens[j + 1]

        if mer == 'PM' and '-' in tr:
            start, end = tr.split('-')

            # normalise "830" -> "8:30"
            start_norm = normalize_time(re.sub(r'(AM|PM)$', '', start))
            end_norm = normalize_time(re.sub(r'(AM|PM)$', '', end))

            # only apply if neither side has AM/PM already
            if not start.endswith(tuple(MERIDIEMS)) and not end.endswith(tuple(MERIDIEMS)):
                # if the end hour is typical finishing time (4 or 6), assume day pattern
                try:
                    end_hour = int(end_norm.split(':')[0])
                except ValueError:
                    end_hour = None

                if end_hour in (4, 6):
                    start = start_norm + "AM"
                    end = end_norm + "PM"
                    normalized_tokens[j] = f"{start}-{end}"
                    del normalized_tokens[j + 1]
                    continue

        j += 1

    # --- 3. Day-range fixes: 'FRI I MON' or 'THURSDAY I MONDAY' -> 'MON-FRI' (with flag) ---
    k = 0
    while k < len(normalized_tokens) - 2:
        d1, mid, d2 = normalized_tokens[k:k + 3]

        c1 = canonical_day(d1)
        c2 = canonical_day(d2)

        if c1 and c2 and mid in {'-', 'I'}:
            original_range = f"{d1}-{d2}"

            # treat "I" as a dash in this context only
            if mid == 'I':
                corrections['I'] = '-'

            idx1 = DAYS_ORDER.index(c1)
            idx2 = DAYS_ORDER.index(c2)

            # decide whether we think this is "backwards" (wrap-around)
            # idx1 > idx2 => like FRI-MON, SAT-TUE, etc.
            if idx1 > idx2:
                # We *still* reverse to make parsing easier,
                # but we explicitly flag that this was a wrap-style pair.
                corrections.setdefault('reversed_day_ranges', []).append({
                    "original": original_range,
                    "canonical": f"{c2}-{c1}",
                    "wrap": True,   # means c1..c2 crosses Sunday in natural order
                })
                c1, c2 = c2, c1
            else:
                # only record if something *changed*
                if f"{d1}-{d2}" != f"{c1}-{c2}":
                    corrections.setdefault('reversed_day_ranges', []).append({
                        "original": original_range,
                        "canonical": f"{c1}-{c2}",
                        "wrap": False,
                    })

            # overwrite tokens with canonical forms and normalized dash
            normalized_tokens[k] = c1
            normalized_tokens[k + 1] = '-'
            normalized_tokens[k + 2] = c2

        k += 1


    # join 'MON - FRI' into 'MON-FRI'
    j = 0
    joined = []
    while j < len(normalized_tokens):
        if j + 2 < len(normalized_tokens) \
                and normalized_tokens[j] in DAYS_ORDER \
                and normalized_tokens[j + 1] == '-' \
                and normalized_tokens[j + 2] in DAYS_ORDER:
            joined.append(f"{normalized_tokens[j]}-{normalized_tokens[j + 2]}")
            j += 3
        else:
            joined.append(normalized_tokens[j])
            j += 1
    normalized_tokens = joined
    
    # --- 3b. Canonicalise standalone day tokens to 3-letter form ---
    for idx, tok in enumerate(normalized_tokens):
        # skip already formed ranges like MON-FRI
        if '-' in tok:
            continue
        c = canonical_day(tok)
        if c:
            normalized_tokens[idx] = c

    # --- 3c. Join "TUE FRI" into "TUE-FRI" (no connector case) ---
    joined_days = []
    i = 0
    while i < len(normalized_tokens):
        if i + 1 < len(normalized_tokens):
            c1 = canonical_day(normalized_tokens[i])
            c2 = canonical_day(normalized_tokens[i + 1])
        else:
            c1 = c2 = None

        if c1 and c2:
            original_range = f"{normalized_tokens[i]}-{normalized_tokens[i + 1]}"
            idx1 = DAYS_ORDER.index(c1)
            idx2 = DAYS_ORDER.index(c2)

            wrap = idx1 > idx2
            if wrap:
                # e.g. FRI MON -> MON-FRI but mark wrap
                corrections.setdefault('reversed_day_ranges', []).append({
                    "original": original_range,
                    "canonical": f"{DAYS_ORDER[idx2]}-{DAYS_ORDER[idx1]}",
                    "wrap": True,
                })
                c_start, c_end = DAYS_ORDER[idx2], DAYS_ORDER[idx1]
            else:
                c_start, c_end = c1, c2
                if c1 != c_start or c2 != c_end:
                    corrections.setdefault('reversed_day_ranges', []).append({
                        "original": original_range,
                        "canonical": f"{c_start}-{c_end}",
                        "wrap": False,
                    })

            joined_days.append(f"{c_start}-{c_end}")
            i += 2
        else:
            joined_days.append(normalized_tokens[i])
            i += 1

    normalized_tokens = joined_days     

    # --- 3d. Split non-day hyphenated tokens into components for better correction ---
    split_tokens = []
    for tok in normalized_tokens:
        if '-' in tok and not is_time_range(tok) and not canonical_day(tok) and tok not in vocab:
            # e.g. "N-PMBLIC" -> ["N", "PMBLIC"]
            parts = tok.split('-')
            # keep only non-empty parts
            for p in parts:
                if p:
                    split_tokens.append(p)
        else:
            split_tokens.append(tok)
    normalized_tokens = split_tokens

    # --- 4. Spell-correct safely ---
    for idx, token in enumerate(normalized_tokens):
        # skip things we don't want to touch
        if canonical_day(token):
            # day tokens are handled via canonical_day, not via spell-correct
            continue
        if token in vocab:
            continue
        if token in {'AM', 'PM'}:
            continue
        if token.isdigit():
            continue
        if any(ch.isdigit() for ch in token):
            continue
        if '-' in token and token not in vocab:
            # don't aggressively correct ranges like MON-FRI here
            continue
        if token == "UN":
            token = "SUN"
            normalized_tokens[idx] = "SUN"
            continue
        if token == "SVT":
            token = "SAT"
            normalized_tokens[idx] = "SAT"
            continue
        if len(token) <= 2:
            continue  # too short, unsafe

        closest = get_close_matches(token, vocab, n=1, cutoff=0.7)
        if closest:
            candidate = closest[0]
            dist = levenshtein(token, candidate)
            if dist <= max_dist:
                corrections[token] = candidate
                normalized_tokens[idx] = candidate

    # --- Fix "ON STOPPING" / "ON STOP" OCR mixups to "NO STOPPING" / "NO STOP" ---

    # 1) Token-level fixes
    for i in range(len(normalized_tokens) - 1):
        # If we see ON/0N/N0 immediately before STOP or STOPPING, make it NO
        if normalized_tokens[i] in {"ON", "0N", "N0"} and normalized_tokens[i+1] in {"STOP", "STOPPING"}:
            normalized_tokens[i] = "NO"

    # keep your existing swap but broaden the right hand token set
    for i in range(len(normalized_tokens) - 1):
        if normalized_tokens[i] in {"STOP", "STOPPING"} and normalized_tokens[i+1] in {"NO", "ON", "0N", "N0"}:
            # swap to "NO STOP(PING)"
            left = "NO"  # normalize ON/0N/N0 to NO
            right = normalized_tokens[i]  # STOP or STOPPING
            normalized_tokens[i], normalized_tokens[i+1] = left, right

    # 2) String-level fallback (handles odd spacing/hyphens)
    joined_text = " ".join(normalized_tokens)
    # Accept 'ON' or '0N' before STOP/STOPPING
    joined_text = re.sub(r"\b[0O]N\s+STOPPING\b", "NO STOPPING", joined_text)
    joined_text = re.sub(r"\b[0O]N\s+STOP\b",     "NO STOP",     joined_text)

    # re-tokenize to keep pipeline consistent
    normalized_tokens = joined_text.split()
    
    # --- FINAL PASS 0: handle "830 12 30 PM AM" -> "8:30AM-12:30PM" ---
    final_tokens0 = []
    i = 0
    while i < len(normalized_tokens):
        if (
            i + 4 < len(normalized_tokens)
            and re.fullmatch(r"\d{3,4}", normalized_tokens[i])         # "830"
            and normalized_tokens[i + 1].isdigit()                    # "12"
            and normalized_tokens[i + 2].isdigit()                    # "30"
            and len(normalized_tokens[i + 2]) == 2
            and normalized_tokens[i + 3] in MERIDIEMS                 # "PM"
            and normalized_tokens[i + 4] in MERIDIEMS                 # "AM"
        ):
            t1   = normalized_tokens[i]
            h2   = normalized_tokens[i + 1]
            m2   = normalized_tokens[i + 2]
            merA = normalized_tokens[i + 3]
            merB = normalized_tokens[i + 4]

            # normalize start time: 830 -> 8:30
            start = normalize_time(t1)

            # build end time: 12 + 30 -> 12:30
            end = f"{int(h2)}:{m2}"

            # ensure AM -> PM ordering regardless of original order
            meridiems = sorted([merA, merB], key=lambda x: ('AM', 'PM').index(x))
            start += meridiems[0]   # AM
            end   += meridiems[1]   # PM

            final_tokens0.append(f"{start}-{end}")  # "8:30AM-12:30PM"
            i += 5
            continue

        final_tokens0.append(normalized_tokens[i])
        i += 1

    normalized_tokens = final_tokens0
    
    # --- FINAL PASS 1: join simple time ranges 8 AM 6 PM -> 8AM-6PM ---
    final_tokens = []
    i = 0
    while i < len(normalized_tokens):
        if (
            i + 3 < len(normalized_tokens)
            and normalized_tokens[i].isdigit()
            and 1 <= int(normalized_tokens[i]) <= 12
            and normalized_tokens[i + 1] in MERIDIEMS
            and normalized_tokens[i + 2].isdigit()
            and 1 <= int(normalized_tokens[i + 2]) <= 12
            and normalized_tokens[i + 3] in MERIDIEMS
        ):
            start = normalized_tokens[i] + normalized_tokens[i + 1]   # e.g. 8 + AM
            end   = normalized_tokens[i + 2] + normalized_tokens[i + 3]  # 6 + PM
            final_tokens.append(f"{start}-{end}")                     # 8AM-6PM
            i += 4
        else:
            final_tokens.append(normalized_tokens[i])
            i += 1

    normalized_tokens = final_tokens

    # --- FINAL PASS 2: join bare day ranges MON FRI -> MON-FRI ---
    final_days = []
    i = 0
    while i < len(normalized_tokens):
        if i + 1 < len(normalized_tokens):
            d1 = canonical_day(normalized_tokens[i])
            d2 = canonical_day(normalized_tokens[i + 1])
        else:
            d1 = d2 = None

        if d1 and d2:
            # both are valid days, join into range
            final_days.append(f"{d1}-{d2}")
            i += 2
        else:
            final_days.append(normalized_tokens[i])
            i += 1

    normalized_tokens = final_days
    
    # --- FINAL PASS 3: fix "[DAY] I [DAY]" -> "DAY1-DAY2" ---
    fixed_days_I = []
    i = 0
    while i < len(normalized_tokens):
        if i + 2 < len(normalized_tokens):
            d1 = canonical_day(normalized_tokens[i])
            mid = normalized_tokens[i + 1]
            d2 = canonical_day(normalized_tokens[i + 2])
        else:
            d1 = d2 = mid = None

        if d1 and d2 and mid == "I":
            # optional: keep wrap logic if you care about FRI-MON vs MON-FRI
            idx1 = DAYS_ORDER.index(d1)
            idx2 = DAYS_ORDER.index(d2)
            if idx1 <= idx2:
                start_day, end_day = d1, d2
            else:
                # wrap-style pair like FRI I MON -> MON-FRI
                start_day, end_day = d2, d1

            fixed_days_I.append(f"{start_day}-{end_day}")
            i += 3
        else:
            fixed_days_I.append(normalized_tokens[i])
            i += 1

    normalized_tokens = fixed_days_I
    
    # --- FINAL PASS 4: canonicalise any remaining day tokens to 3-letter form ---
    canonicalised = []
    for tok in normalized_tokens:
        if "-" in tok:
            parts = tok.split("-")
            canon_parts = []
            all_days = True
            for p in parts:
                c = canonical_day(p)
                if c:
                    canon_parts.append(c)
                else:
                     all_days = False
                     break
            if all_days and len(canon_parts) == 2:
                new_tok = f"{canon_parts[0]}-{canon_parts[1]}"
                if new_tok != tok:
                    corrections.setdefault("day_canonical_ranges", []).append({
                        "original": tok,
                        "canonical": new_tok,
                    })
                canonicalised.append(new_tok)
            else:
                canonicalised.append(tok)
        else:
            c = canonical_day(tok)
            if c and c != tok:
                corrections.setdefault("day_canonical", []).append({
                    "original": tok,
                    "canonical": c,
                })
                canonicalised.append(c)
            else:
                canonicalised.append(tok)

    normalized_tokens = canonicalised

    # now safely define normalized before returning
    normalized = " ".join(normalized_tokens)

    # collapse " - " between tokens like "8AM - 6PM" or "8AM - 1PM"
    # but not "2P - 10AM"
    normalized = re.sub(r'\b([A-Z0-9]+)\s*-\s*(\d[A-Z]+)\b', r'\1-\2', normalized)
    # existing rules
    normalized = re.sub(r'\b([A-Z]+)\s*-\s*([A-Z]+)\b', r'\1-\2', normalized)
    normalized = re.sub(r'\bM\s*-\s*(\d+)\b', r'M-\1', normalized)
    
    # split stuck AREA tokens like AREA15
    normalized = re.sub(r"\bAREA([A-Z0-9]+)\b", r"AREA \1", normalized)
    
    return {
        "raw": raw,
        "normalized": normalized,
        "tokens": normalized_tokens,
        "clean_text": normalized,
        "corrections": corrections,
    }

In [ ]:
examples = [
    "2P -10 8 AM PM PERMIT HOLDERS EXCEPTED AREA 22 AREA M1",
    "4P 8AM - MIDNIGHT MON-FRI 8AM -1 PM SAT PERMIT HOLDERS EXCEPTED AREA15",
    "2P 9Am-6 PM MON - SAT & HOLIDAYS PUBLIC PERMIT HOLDERS EXCEPTED AREA 13",
    "WORKS ZONE 7 30 5 30 AM PM MON- FRI 7 30 - 3 30 AM PM SAT"
]

for s in examples:
    cleaned = clean_ocr_text(s, vocab=DOMAIN_VOCAB, max_dist=2)
    print("RAW:       ", s)
    print("NORMALIZED:", cleaned["normalized"])
    print("CLEAN:     ", cleaned["clean_text"])
    print("CORRS:     ", cleaned["corrections"])
    print("-" * 50)

In [ ]:
import json
from pathlib import Path

def clean_results_file(input_path: str, output_path: str, vocab, max_dist: int = 2):
    """
    Load an OCR results JSON file, clean each merged_text with clean_ocr_text,
    and write out an augmented JSON file with normalized/clean fields.
    """
    input_path = Path(input_path)
    output_path = Path(output_path)

    with input_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    for img_name, entry in data.items():
        merged = entry.get("merged_text", "") or ""
        cleaned = clean_ocr_text(merged, vocab=vocab, max_dist=max_dist)

        entry["normalized_text"] = cleaned["normalized"]
        entry["clean_text"] = cleaned["clean_text"]
        entry["clean_tokens"] = cleaned["tokens"]
        entry["token_corrections"] = cleaned["corrections"]

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Done cleaning OCR text: {input_path.name} -> {output_path.name}")

In [ ]:
# run for validation and test sets
clean_results_file(
    "ocr/results_val_with_gt.json",
    "ocr/results_val_cleaned.json",
    vocab=DOMAIN_VOCAB,
    max_dist=2,
)

clean_results_file(
    "ocr_test/results_test_with_gt.json",
    "ocr_test/results_test_cleaned.json",
    vocab=DOMAIN_VOCAB,
    max_dist=2,
)

### JSON Structured Extraction

In [ ]:
import json
import re
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

# 1. Time + day helpers
DAY_ORDER = ["MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN"]


def parse_time_token(tok: str) -> Optional[str]:
    """
    Convert a time token to 24h 'HH:MM'.
    Handles:
      - '8AM', '8PM', '8:30AM', '12AM', '12PM'
      - '8', '08', '18:00' (24h-style)
    """
    tok = tok.strip().upper()
    
    # Special cases
    if tok == "MIDNIGHT":
        return "00:00"
    if tok == "NOON":
        return "12:00"
    
    # 1) 12h with AM/PM
    m = re.match(r'^(\d{1,2})(?::(\d{2}))?(AM|PM)$', tok)
    if m:
        h = int(m.group(1))
        mnt = int(m.group(2) or 0)
        mer = m.group(3)

        if mer == "AM":
            if h == 12:
                h = 0
        else:  # PM
            if h != 12:
                h += 12

        if 0 <= h <= 23 and 0 <= mnt <= 59:
            return f"{h:02d}:{mnt:02d}"
        return None

    # 2) 24h-style 'H:MM' or 'HH:MM'
    m = re.match(r'^(\d{1,2}):(\d{2})$', tok)
    if m:
        h = int(m.group(1))
        mnt = int(m.group(2))
        if 0 <= h <= 23 and 0 <= mnt <= 59:
            return f"{h:02d}:{mnt:02d}"
        return None

    # 3) Plain hour
    if tok.isdigit():
        h = int(tok)
        if 0 <= h <= 23:
            return f"{h:02d}:00"

    return None


TIME_RANGE_RE = re.compile(
    r'(\d{1,2}(?::\d{2})?(?:AM|PM)?|MIDNIGHT|NOON)\s*-\s*'
    r'(\d{1,2}(?::\d{2})?(?:AM|PM)?|MIDNIGHT|NOON)',
    re.IGNORECASE
)


def get_day_blocks(text: str) -> List[List[str]]:
    """
    Return a list of day-lists in textual order.

    Examples:
      "MON-FRI SAT"   -> [["MON","TUE","WED","THU","FRI"], ["SAT"]]
      "MON-WED SUN"   -> [["MON","TUE","WED"], ["SUN"]]
      "SAT-SUN"       -> [["SAT","SUN"]]
      "MON TUE WED"   -> [["MON","TUE","WED"]]  (falls back to one block)
    """
    t = text.upper()
    blocks: List[List[str]] = []

    # first try to use explicit ranges/single days via DAY_RANGE_RE
    matches = list(DAY_RANGE_RE.finditer(t))
    for m in matches:
        d1_raw = m.group(1)
        d2_raw = m.group(2)
        d1 = canonical_day(d1_raw) if d1_raw else None
        d2 = canonical_day(d2_raw) if d2_raw else None

        if not d1:
            continue

        if d2:
            days = expand_day_range(d1, d2)
        else:
            days = [d1]
        blocks.append(days)

    if blocks:
        return blocks

    # fallback: no explicit ranges matched; treat all days as one block
    all_days = parse_days_for_block(text)
    if all_days:
        return [all_days]

    return []


def parse_time_range(range_str: str) -> Optional[Tuple[str, str]]:
    """
    Parse time ranges with some intelligence about missing AM/PM:
      '7AM-4'           -> ('07:00','16:00')  # infer 4PM
      '7AM-4PM'         -> ('07:00','16:00')
      '7-16'            -> ('07:00','16:00')  # treat as 24h
      '8:30AM-6:00PM'   -> ('08:30','18:00')
      '06:00-18:00'     -> ('06:00','18:00')
    """
    m = TIME_RANGE_RE.search(range_str)
    if not m:
        return None

    start_raw, end_raw = m.group(1).upper(), m.group(2).upper()

    
    def parse_raw(tok: str):
        """
        Parse a side of the range into (hour, minute, meridiem).
        Enforces:
          - if meridiem present -> 1–12
          - if no meridiem      -> 0–23
          - minutes             -> 0–59
        Returns None if invalid.
        """
        m2 = re.match(r'^(\d{1,2})(?::(\d{2}))?(AM|PM)?$', tok)
        if not m2:
            return None
        h = int(m2.group(1))
        mnt = int(m2.group(2) or 0)
        mer = m2.group(3)  # 'AM', 'PM' or None

        # minute bounds
        if not (0 <= mnt <= 59):
            return None

        if mer in ("AM", "PM"):
            # 12h clock must be 1–12
            if not (1 <= h <= 12):
                return None
        else:
            # 24h clock: 0–23
            if not (0 <= h <= 23):
                return None

        return h, mnt, mer

    
    def to_24(h: int, mnt: int, mer: Optional[str]) -> Optional[str]:
        """
        Convert (hour, minute, meridiem) to 'HH:MM' with bounds.
        Returns None if invalid.
        """
        if not (0 <= mnt <= 59):
            return None

        if mer in ("AM", "PM"):
            if not (1 <= h <= 12):
                return None
            if mer == "AM":
                if h == 12:
                    h = 0
            else:  # PM
                if h != 12:
                    h += 12
        else:
            # treat as 24h hour
            if not (0 <= h <= 23):
                return None

        return f"{h:02d}:{mnt:02d}"

    s = parse_raw(start_raw)
    e = parse_raw(end_raw)

    # fallback: if pattern fails, use simple token parsing
    if not s or not e:
        start_24 = parse_time_token(start_raw)
        end_24 = parse_time_token(end_raw)
        if not start_24 or not end_24:
            return None
        return start_24, end_24

    sh, sm, s_mer = s
    eh, em, e_mer = e

    # Case 1: both sides already have AM/PM
    if s_mer and e_mer:
        start_24 = to_24(sh, sm, s_mer)
        end_24 = to_24(eh, em, e_mer)
        if not start_24 or not end_24:
            return None
        return start_24, end_24

    # Case 2: start has meridiem, end missing -> infer end meridiem
    if s_mer and not e_mer:
        start_24 = to_24(sh, sm, s_mer)
        if not start_24:
            return None

        # try same meridiem first
        end_24_same = to_24(eh, em, s_mer)
        if end_24_same and end_24_same >= start_24:
            return start_24, end_24_same

        # flip if backward
        opp = "PM" if s_mer == "AM" else "AM"
        end_24_opp = to_24(eh, em, opp)
        if not end_24_opp:
            return None
        return start_24, end_24_opp

    # Case 3: end has meridiem, start missing
    if e_mer and not s_mer:
        end_24 = to_24(eh, em, e_mer)
        if not end_24:
            return None

        start_24_same = to_24(sh, sm, e_mer)
        if start_24_same and start_24_same <= end_24:
            return start_24_same, end_24

        opp = "PM" if e_mer == "AM" else "AM"
        start_24_opp = to_24(sh, sm, opp)
        if not start_24_opp:
            return None
        return start_24_opp, end_24

    # Case 4: neither side has AM/PM -> treat as 24h, but still bounded
    start_24 = parse_time_token(start_raw)
    end_24 = parse_time_token(end_raw)
    if not start_24 or not end_24:
        return None

    # Heuristic: if both hours are < 12 and end < start, assume end is PM
    try:
        sh, sm = map(int, start_24.split(":"))
        eh, em = map(int, end_24.split(":"))
        if sh < 12 and eh < 12 and (eh < sh):
            eh += 12
            end_24 = f"{eh:02d}:{em:02d}"
    except Exception:
        # if parsing fails, just keep as-is
        pass

    return start_24, end_24


def expand_day_range(start: str, end: str) -> list[str]:
    start = start.upper()
    end = end.upper()
    if start not in DAY_ORDER or end not in DAY_ORDER:
        return []
    i1, i2 = DAY_ORDER.index(start), DAY_ORDER.index(end)
    if i1 <= i2:
        return DAY_ORDER[i1:i2+1]
    # wrap-around (e.g. FRI-MON)
    return DAY_ORDER[i1:] + DAY_ORDER[:i2+1]


DAY_RANGE_RE = re.compile(
    r'\b(MON|TUE|WED|THU|FRI|SAT|SUN)'
    r'(?:-(MON|TUE|WED|THU|FRI|SAT|SUN))?\b',
    re.IGNORECASE
)

def parse_days_for_block(block: str) -> List[str]:
    days: List[str] = []
    for m in DAY_RANGE_RE.finditer(block):
        d1 = m.group(1).upper()
        d2 = m.group(2).upper() if m.group(2) else None
        if d2:
            days.extend(expand_day_range(d1, d2))
        else:
            days.append(d1)

    seen = set()
    out = []
    for d in days:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out

# 2. Payment, time-limit, zones, notes

def detect_payment_type(text: str) -> str:
    t = text.upper()
    if "METER" in t:
        return "METER"
    if "TICKET" in t:
        return "TICKET"
    return "FREE"


def extract_time_limit(text: str, det_categories: List[int]) -> Optional[str]:
    """
    Time limit rules:
      - '1P', '2P', '4P', '1/2P', etc. → use directly
      - bare 'P' -> NOT treated as time limit (even for disabled P ONLY)
      - minutes-based:
           X MINUTE(S), MINUTE(S) X -> 'XMIN'
           '15MIN' -> '15MIN'
    """
    t = text.upper()
    tokens = t.split()

    # 1) P-style with number
    for tok in tokens:
        if re.fullmatch(r'\d+(?:/\d+)?P', tok):
            return tok

    # 2) minutes-based words
    for i, tok in enumerate(tokens):
        if tok in {"MINUTE", "MINUTES"}:
            num = None
            if i > 0 and tokens[i-1].isdigit():
                num = tokens[i-1]
            elif i+1 < len(tokens) and tokens[i+1].isdigit():
                num = tokens[i+1]
            if num:
                return f"{int(num)}MIN"
            else:
                return "MIN"

    # 3) '15MIN' style
    for tok in tokens:
        m = re.fullmatch(r'(\d+)MIN', tok)
        if m:
            return f"{int(m.group(1))}MIN"

    return None


def is_permit_code_token(tok: str) -> bool:
    if not re.fullmatch(r"[A-Z0-9]+", tok):
        return False
    if any(ch.isdigit() for ch in tok):
        return True
    return len(tok) <= 3  # short letter code (e.g. G, M1, AB)


NON_PERMIT_ZONE_PREFIXES = {
    "BUS", "LOADING", "MAIL", "TAXI", "TAXIS",
    "METER", "TICKET", "NO", "DISABLED", "WORKS", "TRUCK"
}


def extract_permit_zone(text: str) -> Optional[str]:
    """
    Extract permit zones via AREA/ZONE patterns, avoiding functional zones:
      - AREA 15, AREA15, 13 AREA  -> 'AREA 15' / 'AREA 13'
      - AREA G AREA 23            -> 'AREA G; AREA 23'
      - ZONE 10 / 10 ZONE / ZONE10
      But not BUS ZONE, LOADING ZONE, MAIL ZONE, TAXI ZONE, etc.
    """
    t = text.upper().replace("-", " ")
    tokens = t.split()
    zones: List[str] = []
    n = len(tokens)

    for i, tok in enumerate(tokens):
        # AREA handling
        if tok == "AREA":
            before = tokens[i-1] if i > 0 else None
            after = tokens[i+1] if i+1 < n else None

            chosen_code = None
            if after and is_permit_code_token(after):
                chosen_code = after
            elif before and is_permit_code_token(before):
                chosen_code = before

            if chosen_code:
                zone = f"AREA {chosen_code}"
            else:
                zone = "AREA"
            zones.append(zone)
            continue

        # AREA15 pattern
        m = re.fullmatch(r"AREA([A-Z0-9]+)", tok)
        if m:
            zones.append(f"AREA {m.group(1)}")
            continue

        # ZONE handling (permit zones only)
        if tok == "ZONE":
            before = tokens[i-1] if i > 0 else None
            after = tokens[i+1] if i+1 < n else None

            if before in NON_PERMIT_ZONE_PREFIXES:
                continue

            chosen_code = None
            if after and is_permit_code_token(after):
                chosen_code = after
            elif before and is_permit_code_token(before):
                chosen_code = before

            if chosen_code:
                zone = f"ZONE {chosen_code}"
            else:
                zone = "ZONE"
            zones.append(zone)
            continue

        # ZONE10 style (avoid BUSZONE etc via previous token)
        m2 = re.fullmatch(r"ZONE([A-Z0-9]+)", tok)
        if m2:
            prev = tokens[i-1] if i > 0 else None
            if prev in NON_PERMIT_ZONE_PREFIXES:
                continue
            zones.append(f"ZONE {m2.group(1)}")
            continue

    seen = set()
    uniq = []
    for z in zones:
        if z not in seen:
            seen.add(z)
            uniq.append(z)

    if not uniq:
        return None
    if len(uniq) == 1:
        return uniq[0]
    return " ".join(uniq)


def detect_sign_notes(text: str) -> List[str]:
    t = text.upper()
    tokens = t.split()
    wset = set(tokens)
    notes = []

    if {"PERMIT", "HOLDERS", "EXCEPTED"}.issubset(wset):
        notes.append("PERMIT HOLDERS EXCEPTED")

    if {"AUTHORISED", "CAR", "SHARE", "VEHICLES", "EXCEPTED"}.issubset(wset):
        notes.append("AUTHORISED CAR SHARE VEHICLES EXCEPTED")

    if {"POLICE", "VEHICLES", "EXCEPTED"}.issubset(wset):
        notes.append("POLICE VEHICLES EXCEPTED")
        
    if {"AUST", "POST", "VEHICLES", "EXCEPTED"}.issubset(wset):
        notes.append("AUST POST VEHICLES EXCEPTED")
        
    if "SPECIAL" in wset and "EVENT" in wset and "CLEARWAY" in wset:
        notes.append("SPECIAL EVENT")

    return notes


def add_note(existing: Optional[str], new_note: str) -> str:
    if not existing:
        return new_note
    if new_note in existing:
        return existing
    return existing + "; " + new_note


# 3. Sign type & rule extraction

def has_paid_or_time_limit_semantics(t: str) -> bool:
    """
    True if the text clearly looks like a timed/paid parking sign
    (1P/2P/etc, MINUTE(S), TICKET, METER, etc.).
    Used to *avoid* misclassifying those as NO PARKING just because
    a no-parking symbol exists somewhere in the image.
    """
    t = t.upper()

    # explicit time-limit "P" styles
    if re.search(r"\b[124]P\b", t):   # 1P, 2P, 4P
        return True

    # minutes-based
    if "MINUTE" in t or "MINUTES" in t:
        return True

    # paid parking hints
    if "TICKET" in t or "METER" in t:
        return True

    return False


def detect_restriction_sign_type(text: str, det_categories) -> str:
    """
    Decide restriction_sign_type using:
      1) strong text-based types (NO STOPPING, CLEARWAY, ZONE types, ANGLE PARKING)
      2) symbol-aware upgrades:
           - DISABLED PARKING if cat 3 present
           - NO PARKING if cat 2 present and text does NOT look like 1P/TICKET/METER/MINUTE
    """
    t = text.upper()
    cats = set(det_categories)

    # strong text-based classes
    if "STOPPING" in t:
        return "NO STOPPING"

    if "CLEARWAY" in t:
        return "CLEARWAY"

    if re.search(r"\bLOADING\s+ZONE\b", t) or "LOADINGZONE" in t:
        return "LOADING ZONE"

    if re.search(r"\bBUS\s+ZONE\b", t) or "BUSZONE" in t:
        return "BUS ZONE"

    if (
        re.search(r"\bTAXI(S)?\s+ZONE\b", t)
        or "TAXIZONE" in t
        or "TAXISZONE" in t
    ):
        return "TAXI ZONE"

    if re.search(r"\bMAIL\s+ZONE\b", t) or "MAILZONE" in t:
        return "MAIL ZONE"

    if (
        re.search(r"\bWORKS?\s+ZONE\b", t)
        or "WORKZONE" in t
        or "WORKSZONE" in t
    ):
        return "WORKS ZONE"

    if (
        re.search(r"\bTRUCK(S)?\s+ZONE\b", t)
        or "TRUCKZONE" in t
    ):
        return "TRUCK ZONE"

    if "ANGLE" in t and "PARKING" in t:
        return "ANGLE PARKING"

    # base type + symbol-based refinement
    primary = "TIME"
    has_paid_limit = has_paid_or_time_limit_semantics(t)

    # prefer disabled symbol if present at all
    if primary == "TIME" and 3 in cats and not has_paid_limit and "P" in t:
        return "DISABLED PARKING"

    # only if no disabled symbol: consider NO PARKING symbol
    if primary == "TIME" and 2 in cats and not has_paid_limit:
        return "NO PARKING"

    return primary


def extract_rules(text: str, time_limit: Optional[str], payment_type: str) -> List[Dict]:
    """
    Build rule objects from the text by finding time ranges + nearby day blocks.
    """
    t = text.upper()
    rules: List[Dict] = []

    ranges = []
    for m in TIME_RANGE_RE.finditer(t):
        ranges.append({
            "range_str": m.group(0),
            "start": m.start(),
            "end": m.end(),
        })

    for idx, r in enumerate(ranges):
        range_str = r["range_str"]
        end_pos = r["end"]

        if idx + 1 < len(ranges):
            next_start = ranges[idx+1]["start"]
            after_range = t[end_pos:next_start]
        else:
            after_range = t[end_pos:]

        parsed = parse_time_range(range_str)
        if not parsed:
            start_24 = end_24 = None
        else:
            start_24, end_24 = parsed

        days = parse_days_for_block(after_range)

        rules.append({
            "rule_id": idx + 1,
            "time_limit": time_limit,
            "days": days,
            "time_start": start_24,
            "time_end": end_24,
            "payment_type": payment_type,
            "notes": None,
            "_span": (r["start"], r["end"]),
        })

    return rules


def assign_rule_level_notes(text: str, rules: List[Dict]) -> None:
    """
    Attach 'PUBLIC HOLIDAYS' and 'OTHER TIMES' to closest/last rules.
    """
    if not rules:
        return

    t = text.upper()
    tokens = t.split()
    wset = set(tokens)

    # PUBLIC HOLIDAYS
    has_public = "PUBLIC" in wset
    has_holiday = any(w.startswith("HOLIDAY") for w in wset)
    if has_public and has_holiday:
        pub_pos = t.find("PUBLIC")
        if pub_pos != -1:
            best_idx = 0
            best_dist = None
            for i, r in enumerate(rules):
                s, e = r["_span"]
                mid = (s + e) // 2
                dist = abs(pub_pos - mid)
                if best_dist is None or dist < best_dist:
                    best_dist = dist
                    best_idx = i
            rules[best_idx]["notes"] = add_note(
                rules[best_idx].get("notes"),
                "PUBLIC HOLIDAYS"
            )

    # OTHER TIMES -> last rule
    if "OTHER" in wset and "TIMES" in wset:
        rules[-1]["notes"] = add_note(rules[-1].get("notes"), "OTHER TIMES")


def strip_internal_spans(rules: List[Dict]) -> None:
    for r in rules:
        r.pop("_span", None)


# 4. Extract one crop

def extract_sign_from_crop(clean_text: str, det_categories: List[int]) -> Dict:
    """
    End-to-end extractor for a single cropped OCR region.
    """
    restriction_type = detect_restriction_sign_type(clean_text, det_categories)
    permit_zone = extract_permit_zone(clean_text)
    payment_type = detect_payment_type(clean_text)
    time_limit = extract_time_limit(clean_text, det_categories)
    sign_notes_list = detect_sign_notes(clean_text)
    sign_notes = " ".join(sign_notes_list) if sign_notes_list else None

    rules = extract_rules(clean_text, time_limit, payment_type)
    assign_rule_level_notes(clean_text, rules)

    if rules and restriction_type != "CLEARWAY":
        strip_internal_spans(rules)
    elif not rules and restriction_type != "CLEARWAY":
        # when time parsing fails, salvage what we can from day structure.
        day_blocks = get_day_blocks(clean_text)

        if day_blocks:
            rules = []
            rule_id = 1
            for block_days in day_blocks:
                rules.append({
                    "rule_id": rule_id,
                    "time_limit": time_limit,
                    "days": block_days,
                    "time_start": None,
                    "time_end": None,
                    "payment_type": payment_type,
                    "notes": None,
                })
                rule_id += 1
        else:
            # last resort: no days found, keep previous behaviour
            rules = [{
                "rule_id": 1,
                "time_limit": time_limit,
                "days": [],
                "time_start": None,
                "time_end": None,
                "payment_type": payment_type,
                "notes": None,
            }]

    sign = {
        "sign_id": 1,
        "sign_type": "RESTRICTION",
        "restriction_sign_type": restriction_type,
        "permit_zone": permit_zone,
        "notes": sign_notes,
        "rules": rules,
    }
    
    t_upper = clean_text.upper()

    if restriction_type == "TIME" and len(rules) == 1:
        # use day blocks in textual order.
        day_blocks = get_day_blocks(clean_text)

        # If there are 2+ distinct blocks, split the single rule accordingly:
        #   - first block keeps the parsed times
        #   - later blocks become extra rules with times = None
        if len(day_blocks) >= 2:
            base = rules[0]
            base["days"] = day_blocks[0]

            next_id = base.get("rule_id", 1) + 1
            for extra_days in day_blocks[1:]:
                rules.append({
                    "rule_id": next_id,
                    "time_limit": base.get("time_limit"),
                    "days": extra_days,
                    "time_start": None,
                    "time_end": None,
                    "payment_type": base.get("payment_type", "FREE"),
                    "notes": None,
                })
    
    # 1) SPECIAL EVENT CLEARWAY → no rules, just a sign-level note
    if sign["restriction_sign_type"] == "CLEARWAY" and "SPECIAL" in t_upper and "EVENT" in t_upper:
        if sign.get("notes") is None:
            sign["notes"] = "SPECIAL EVENT"
        sign["rules"] = []

    # 2) Normal CLEARWAY with no time ranges but with days → create day-only rule
    elif sign["restriction_sign_type"] == "CLEARWAY" and not sign["rules"]:
        day_list = parse_days_for_block(clean_text)
        if day_list:
            sign["rules"] = [{
                "rule_id": 1,
                "time_limit": None,          # clearway typically no P-style limit
                "days": day_list,           # e.g. ["MON", "TUE", ..., "FRI"]
                "time_start": None,
                "time_end": None,
                "payment_type": payment_type,
                "notes": None,
            }]
    
    return {
        "clean_text": clean_text,
        "signs": [sign],
    }


# 5. Build structured predictions for a whole split

def load_detection_categories(predictions_path: str, score_thresh: float = 0.5):
    """
    Returns:
      img_to_cats: dict[str, list[int]]
        e.g. {"IMG_019": [1,2,4,4], ...}
    Grouped by file_name stem (IMG_019 from 'IMG_019.jpg').
    """
    with open(predictions_path, "r", encoding="utf-8") as f:
        preds = json.load(f)

    img_to_cats = defaultdict(list)

    for det in preds:
        file_name = det.get("file_name")  # e.g. "IMG_019.jpg"
        if not file_name:
            continue
        base_id = Path(file_name).stem  # "IMG_019"

        score = det.get("score", 1.0)
        if score < score_thresh:
            continue

        cat = int(det.get("category_id", -1))
        if cat > 0:
            img_to_cats[base_id].append(cat)

    return img_to_cats


def build_structured_predictions(
    ocr_cleaned_path: str,
    predictions_path: str,
    output_path: str
):
    """
    Main driver:
      - load OCR cleaned results (results_val_cleaned.json)
      - load detection predictions (runs/detect/val/predictions.json)
      - for each crop, build structured prediction JSON
      - write to output_path
    """
    # load OCR cleaned (per crop)
    with open(ocr_cleaned_path, "r", encoding="utf-8") as f:
        ocr_data = json.load(f)

    # load detection categories per original image
    img_to_cats = load_detection_categories(predictions_path, score_thresh=0.5)

    pred_struct: Dict[str, Dict] = {}

    for crop_name, entry in ocr_data.items():
        # get clean text: prefer 'clean_text', fall back to 'normalized_text' or 'merged_text'
        clean_text = (
            entry.get("clean_text")
            or entry.get("normalized_text")
            or entry.get("merged_text")
            or ""
        )

        # derive original image id (e.g., 'IMG_012' from 'IMG_012_crop0.png')
        stem = Path(crop_name).stem
        if "_crop" in stem:
            base_id = stem.split("_crop")[0]
        else:
            base_id = stem

        det_categories = img_to_cats.get(base_id, set())

        pred_struct[crop_name] = extract_sign_from_crop(clean_text, det_categories)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(pred_struct, f, indent=2, ensure_ascii=False)

    print(f"Saved structured predictions to {output_path}")

In [ ]:
# for validation
ocr_cleaned_path = "ocr/results_val_cleaned.json"
predictions_path = "runs/detect/val/predictions.json"
output_path = "ocr/structured_val_pred.json"

build_structured_predictions(ocr_cleaned_path, predictions_path, output_path)

In [ ]:
# # for test (skipping test)
# ocr_cleaned_path = "ocr_test/results_test_cleaned.json"
# predictions_path = "runs/detect/val2/predictions.json"
# output_path = "ocr_test/structured_test_pred.json"

# build_structured_predictions(ocr_cleaned_path, predictions_path, output_path)

In [ ]:
def add_symbol_only_no_parking_signs(
    structured_pred_path: str,
    predictions_path: str,
    output_path: str,
    score_thresh: float = 0.5,
):
    # 1) Load existing structured predictions (per crop)
    with open(structured_pred_path, "r", encoding="utf-8") as f:
        preds = json.load(f)

    # 2) Build mapping from base_id -> [crop_keys]
    base_to_crop_keys = defaultdict(list)
    for crop_key in preds.keys():
        stem = Path(crop_key).stem  # e.g. "IMG_019_crop0"
        if "_crop" in stem:
            base_id = stem.split("_crop")[0]
        else:
            base_id = stem
        base_to_crop_keys[base_id].append(crop_key)

    # 3) Load detection categories (including no-parking symbols)
    img_to_cats = load_detection_categories(predictions_path, score_thresh=score_thresh)

    # 4) For each image, figure out how many symbol-only NO PARKING signs we need
    for base_id, cats in img_to_cats.items():
        num_cat2 = sum(1 for c in cats if c == 2)  # no-parking symbol count
        if num_cat2 == 0:
            continue

        # Count existing NO PARKING signs for this base image
        existing_no_parking = 0
        for crop_key in base_to_crop_keys.get(base_id, []):
            entry = preds.get(crop_key, {})
            for sign in entry.get("signs", []):
                if sign.get("restriction_sign_type") == "NO PARKING":
                    existing_no_parking += 1

        # how many extra symbol-only signs do we need?
        extra_needed = max(0, num_cat2 - existing_no_parking)
        if extra_needed == 0:
            continue

        # create extra synthetic entries
        for i in range(extra_needed):
            symbol_key = f"{base_id}_symbol{i}"
            if symbol_key in preds:
                # Unlikely, but avoid overwriting
                continue

            preds[symbol_key] = {
                "clean_text": "",
                "signs": [
                    {
                        "sign_id": 1,
                        "sign_type": "RESTRICTION",
                        "restriction_sign_type": "NO PARKING",
                        "permit_zone": None,
                        "notes": "",
                        "rules": [
                            {
                                "rule_id": 1,
                                "time_limit": None,
                                "days": [],
                                "time_start": None,
                                "time_end": None,
                                "payment_type": "FREE",
                                "notes": None,
                            }
                        ],
                    }
                ],
            }

    # 5) write out the augmented predictions
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(preds, f, indent=2, ensure_ascii=False)

    print(f"Saved augmented predictions (with symbol-only NO PARKING) to {output_path}")

In [ ]:
# example usage for VAL
structured_pred_path = "ocr/structured_val_pred.json"
predictions_path = "runs/detect/val/predictions.json"
output_path = "ocr/structured_val_pred_with_symbols.json"

add_symbol_only_no_parking_signs(
    structured_pred_path,
    predictions_path,
    output_path,
    score_thresh=0.5,
)

### Evaluation - Exact Match
Only evaluate crops that exist in both GT and prediction and if it contains any text (the no parking symbol signs will be skipped as part of the evaluation).

In [ ]:
import json
from collections import defaultdict
import pandas as pd


def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def normalize_nil(x):
    if x in ["", "null", None]:
        return None
    return x


def rules_equal(r1, r2):
    keys = ["time_limit", "time_start", "time_end", "payment_type"]
    for k in keys:
        if normalize_nil(r1.get(k)) != normalize_nil(r2.get(k)):
            return False

    if sorted(r1.get("days", [])) != sorted(r2.get("days", [])):
        return False

    if normalize_nil(r1.get("notes")) != normalize_nil(r2.get("notes")):
        return False

    return True


def evaluate(gt, pred):
    correct = defaultdict(int)
    totals = defaultdict(int)
    rule_correct = 0
    rule_total = 0

    for crop, gt_entry in gt.items():
        if crop not in pred:
            continue
        if not gt_entry.get("signs"):  # skip GT without signs
            continue

        gt_sign = gt_entry["signs"][0]
        pr_sign = pred[crop]["signs"][0]

        # sign-level fields
        fields = ["sign_type", "restriction_sign_type", "permit_zone", "notes"]
        for field in fields:
            gt_val = normalize_nil(gt_sign.get(field))
            pr_val = normalize_nil(pr_sign.get(field))

            totals[field] += 1
            if gt_val == pr_val:
                correct[field] += 1

        # rules
        gt_rules = gt_sign.get("rules", [])
        pr_rules = pr_sign.get("rules", [])

        def sort_key(r):
            d = r.get("days", [])
            first = d[0] if d else ""
            return (first, r.get("time_start") or "", r.get("time_limit") or "")

        gt_sorted = sorted(gt_rules, key=sort_key)
        pr_sorted = sorted(pr_rules, key=sort_key)

        for i in range(min(len(gt_sorted), len(pr_sorted))):
            if rules_equal(gt_sorted[i], pr_sorted[i]):
                rule_correct += 1

        rule_total += len(gt_sorted)

    # construct output table
    rows = []
    for field in ["sign_type", "restriction_sign_type", "permit_zone", "notes"]:
        acc = correct[field] / totals[field] if totals[field] else 0.0
        rows.append({
            "Field": field,
            "Correct": correct[field],
            "Total": totals[field],
            "Accuracy": round(acc, 3)
        })

    acc_rules = rule_correct / rule_total if rule_total else 0.0
    rows.append({
        "Field": "rules",
        "Correct": rule_correct,
        "Total": rule_total,
        "Accuracy": round(acc_rules, 3)
    })

    df = pd.DataFrame(rows)
    return df

In [ ]:
# load files
gt = load_json("ocr/structured_val_gt.json")
pred = load_json("ocr/structured_val_pred_with_symbols.json")

df_results = evaluate(gt, pred)
df_results

| Field | Correct | Total | Accuracy |
| ----- | ----: | --: | ----: | 
| sign_type |   151 | 151 |   1.000 |
| restriction_sign_type |    139 |  151 |    0.921 |
| permit_zone |    146 |   151 |    0.967 |
| notes |   147 | 151 |   0.974 |
| rules |   103 | 195 |   0.528 |

## Demo

In [ ]:
from pathlib import Path
from typing import Dict, Any, List
import json
from PIL import Image

from ultralytics import YOLO
from paddleocr import PaddleOCR

BEST_MODEL_PATH = "runs/detect/train/weights/best.pt"
PARKING_SIGN_CLASS_ID = 0
NO_PARKING_SYMBOL_CLASS_ID = 1
DISABLED_SYMBOL_CLASS_ID = 2
TEXT_REGION_CLASS_ID = 3
DEMO_OCR_MAX_DIST = 2  # Levenshtein distance for clean_ocr_text

yolo_model = YOLO(BEST_MODEL_PATH)
demo_ocr = PaddleOCR(lang="en", use_textline_orientation=False)


def run_ocr_on_crop(crop_path: Path) -> str:
    """
    Run PaddleOCR on a single crop and return a merged text string.
    This is the 'merged_text' equivalent for your demo.
    """
    results = demo_ocr.predict(str(crop_path))
    if not results:
        return ""
    page = results[0]
    rec_texts = page.get("rec_texts", []) or []
    merged = " ".join(rec_texts)
    return merged


def detect_and_crop_text_regions(
    image_path: Path,
    crop_dir: Path,
    pad: int = 10,
) -> Dict[str, Any]:
    """
    Run YOLO on a single image, crop TEXT_REGION_CLASS_ID regions to crop_dir,
    and return:
      - crop_paths: list[Path]
      - det_categories: list[int]   # all categories seen in this image

    Crop filenames always start from 1 for each image:
      <stem>_crop1.png, <stem>_crop2.png, ...
    On rerun, these files are overwritten.
    """
    crop_dir.mkdir(parents=True, exist_ok=True)

    img = Image.open(image_path).convert("RGB")
    w, h = img.size

    results = yolo_model(str(image_path))[0]
    boxes = results.boxes

    det_categories: List[int] = []
    crop_paths: List[Path] = []

    if boxes is None:
        return {"crop_paths": crop_paths, "det_categories": det_categories}

    # collect all categories (image-level, like your notebook)
    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i].item())
        det_categories.append(cls_id)

    # filter text boxes, then index from 1..N so crop names are stable
    text_indices = [
        i for i in range(len(boxes))
        if int(boxes.cls[i].item()) == TEXT_REGION_CLASS_ID
    ]

    for crop_idx, box_i in enumerate(text_indices, start=1):
        x1, y1, x2, y2 = boxes.xyxy[box_i].tolist()
        x1 = max(0, int(x1) - pad)
        y1 = max(0, int(y1) - pad)
        x2 = min(w, int(x2) + pad)
        y2 = min(h, int(y2) + pad)

        if x2 <= x1 or y2 <= y1:
            continue

        crop_img = img.crop((x1, y1, x2, y2))
        crop_name = f"{image_path.stem}_crop{crop_idx}.png"
        crop_path = crop_dir / crop_name
        crop_img.save(crop_path)  # overwrites if exists
        crop_paths.append(crop_path)

    return {
        "crop_paths": crop_paths,
        "det_categories": det_categories,
    }


def build_image_json_from_crops(
    image_path: Path,
    crop_paths: List[Path],
    det_categories: List[int],
) -> Dict[str, Any]:
    """
    For a single image:
      - For each crop:
          * OCR -> merged_text
          * clean_ocr_text(...) -> clean_text
          * extract_sign_from_crop(clean_text, det_categories)
      - Consolidate all resulting signs under one image-level JSON.
      - NEW: If YOLO detected more parking-sign boxes than text-based signs,
             create additional symbol-only signs (NO PARKING / DISABLED / NO STOPPING)
             based on symbol counts.
    """
    signs: List[Dict[str, Any]] = []
    next_sign_id = 1


    # 1. Signs from text crops
    for crop_path in crop_paths:
        merged_text = run_ocr_on_crop(crop_path)
        if not merged_text.strip():
            continue

        # use your existing cleaning pipeline:
        cleaned = clean_ocr_text(
            merged_text,
            vocab=DOMAIN_VOCAB,
            max_dist=DEMO_OCR_MAX_DIST,
        )
        clean_text = (
            cleaned.get("clean_text")
            or cleaned.get("normalized_text")
            or cleaned.get("normalized")
            or merged_text
        )

        # use extractor:
        sign_struct = extract_sign_from_crop(clean_text, det_categories)
        for sign in sign_struct.get("signs", []):
            sign = dict(sign)  # shallow copy
            sign["sign_id"] = next_sign_id
            sign["raw_text"] = merged_text
            sign["clean_text"] = clean_text
            sign["source_crop"] = crop_path.name
            signs.append(sign)
            next_sign_id += 1

    # 2. Add symbol-only signs
    # count how many parking-sign boxes YOLO saw
    total_parking_sign_boxes = sum(
        1 for c in det_categories if c == PARKING_SIGN_CLASS_ID
    )

    current_signs = len(signs)
    extra_needed = max(0, total_parking_sign_boxes - current_signs)

    if extra_needed > 0:
        # symbol counts (image-level)
        total_disabled_symbols = sum(
            1 for c in det_categories if c == DISABLED_SYMBOL_CLASS_ID
        )
        total_no_parking_symbols = sum(
            1 for c in det_categories if c == NO_PARKING_SYMBOL_CLASS_ID
        )

        # existing sign type counts
        existing_disabled_signs = sum(
            1 for s in signs
            if s.get("restriction_sign_type") == "DISABLED PARKING"
        )
        existing_no_parking_signs = sum(
            1 for s in signs
            if s.get("restriction_sign_type") == "NO PARKING"
        )


        def make_symbol_only_sign(restriction_type: str) -> Dict[str, Any]:
            nonlocal next_sign_id
            sign = {
                "sign_id": next_sign_id,
                "sign_type": "RESTRICTION",
                "restriction_sign_type": restriction_type,
                "permit_zone": None,
                "notes": None,
                # minimal rule: same structure as your usual rules, but unknown timings
                "rules": [
                    {
                        "rule_id": 1,
                        "time_limit": None,
                        "days": [],
                        "time_start": None,
                        "time_end": None,
                        "payment_type": "FREE",
                        "notes": None,
                    }
                ],
                "raw_text": "",          # no OCR text
                "clean_text": "",
                "source_crop": "SYMBOL ONLY",
            }
            next_sign_id += 1
            return sign

        # prefer to allocate disabled signs first, then no parking, then no stopping.
        while extra_needed > 0:
            if total_disabled_symbols > existing_disabled_signs:
                signs.append(make_symbol_only_sign("DISABLED PARKING"))
                existing_disabled_signs += 1
                extra_needed -= 1
                continue

            if total_no_parking_symbols > existing_no_parking_signs:
                signs.append(make_symbol_only_sign("NO PARKING"))
                existing_no_parking_signs += 1
                extra_needed -= 1
                continue

            # no more unmatched symbols of the major types; stop.
            break

    # 3. Final image-level JSON
    image_result = {
        "image_id": image_path.name,
        "image_path": image_path.as_posix(),
        "num_signs": len(signs),
        "signs": signs,
    }
    return image_result


def run_demo_inference(
    input_path: str,
    output_dir: str = "demo_output",
    crop_subdir: str = "crops",
) -> Dict[str, Dict[str, Any]]:
    """
    High-level entrypoint.

    - If input_path is a single image:
        processes that image, writes demo_output/<stem>.json (overwrites)

    - If input_path is a folder:
        processes all .jpg/.jpeg/.png in that folder, 1 JSON per image
        (each overwritten if it already exists).

    Returns:
        dict[str, dict]: mapping of image_path -> image-level JSON result.
    """
    input_path = Path(input_path)
    output_root = Path(output_dir)
    crop_dir = output_root / crop_subdir
    output_root.mkdir(parents=True, exist_ok=True)

    results: Dict[str, Dict[str, Any]] = {}

    
    def process_one_image(img_path: Path) -> None:
        print(f"\n[demo] Processing image: {img_path.as_posix()}")
        det_info = detect_and_crop_text_regions(img_path, crop_dir=crop_dir)
        crop_paths = det_info["crop_paths"]
        det_categories = det_info["det_categories"]

        if not crop_paths:
            print("[demo] No text crops found for this image.")
        else:
            print(f"[demo] Found {len(crop_paths)} text crops.")

        img_json = build_image_json_from_crops(
            img_path,
            crop_paths=crop_paths,
            det_categories=det_categories,
        )

        # overwrite JSON for this image if it already exists
        out_path = output_root / f"{img_path.stem}.json"
        out_path.write_text(
            json.dumps(img_json, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print(f"[demo] Saved JSON to {out_path.as_posix()}")

        results[img_path.as_posix()] = img_json
        print(json.dumps(img_json, indent=2, ensure_ascii=False))

    if input_path.is_file():
        process_one_image(input_path)
    else:
        for img_path in sorted(input_path.iterdir()):
            if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                process_one_image(img_path)

    return results

In [ ]:
run_demo_inference("demo/Demo_Image.jpg", output_dir="demo_output")

In [ ]:
run_demo_inference("demo/image_folder", output_dir="demo_output")